## Typicality rating experiment — LLaMA 3.1 8B Instruct

In [1]:
import os
# Must be set BEFORE importing transformers, or it will still probe TensorFlow
#os.environ["USE_TF"] = "0"
#os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

import gc
import re
import pandas as pd
import torch
from transformers import pipeline

#MODEL_NAME = "/data1/shared_models/models--meta-llama--Llama-3.1-8B-Instruct/"  # change to "meta-llama/Meta-Llama-3-8B-Instruct" if you need 3.0

2026-07-16 15:01:07.671003: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-16 15:01:07.684095: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784206867.700164    7314 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784206867.704985    7314 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784206867.716964    7314 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/home/muti/miniconda3/envs/testenv/lib/python3.11/site-packages/tensorflow/python/keras/engine/training_arrays_v1.py:37: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.6)
  from scipy.sparse import issparse  # pylint: disable=g-import-not-at-top


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [16]:
# Set the language to run this notebook for.
# Must match the suffix used in your CSV's instance_<LANGUAGE> column,
# e.g. "English", "German", "Spanish"
LANGUAGE = "Spanish"

instance_col = f"instance_{LANGUAGE}"
norm_rating_col = f"norm_rating_{LANGUAGE}"


In [17]:
df = pd.read_csv('combined_prototypes.csv')

# keep only rows with a non-empty instance for the selected language
df = df[df[instance_col].notna() & (df[instance_col].astype(str).str.strip() != "")].reset_index(drop=True)

df.head(5)


,category,concept_en,instance_English,instance_German,instance_Spanish,norm_rating_English,norm_rating_German,norm_rating_Spanish,n_languages,in_multiple_languages
0,animal,ANT,ant,Ameise,Hormiga,0.308662,0.630556,0.521127,3,True
1,animal,BEE,bee,Biene,Abeja,0.309422,0.711111,0.521127,3,True
2,animal,BUTTERFLY,butterfly,Schmetterling,Mariposa,0.298172,0.746667,0.507042,3,True
3,animal,CROW,crow,Krähe,Cuervo,0.499310,0.820556,0.394366,3,True
4,animal,DUCK,duck,Ente,Pato,0.772608,0.936667,0.830986,3,True


In [18]:
def typicality_prompt(obj, category):
    return f"""
You are participating in a psychology experiment.

Is "{obj}" a prototypical example of the category "{category}"?
Respond with ONLY one word: Yes or No.
"""


In [19]:
from transformers import pipeline
import torch
MODEL_NAME = "/data1/shared_models/models--google--gemma-3-12b-it/snapshots/96b6f1eccf38110c56df3a15bffe176da04bfd80//"
#MODEL_NAME = "/data1/shared_models/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659/"

pipe = pipeline(
    "text-generation",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
Some parameters are on the meta device because they were offloaded to the cpu.
Falling back to torch.float32 because loading with the original dtype failed on the target device.
Device set to use cuda:0


In [20]:
import torch
import torch.nn.functional as F

model = pipe.model
tokenizer = pipe.tokenizer

# verify these are single tokens for your tokenizer (see sanity check below)
yes_id = tokenizer.encode("Yes", add_special_tokens=False)[0]
no_id = tokenizer.encode("No", add_special_tokens=False)[0]

print(tokenizer.decode([yes_id]), tokenizer.decode([no_id]))  # sanity check: should print 'Yes No'

def get_typicality(obj, category):
    prompt = typicality_prompt(obj, category)
    messages = [{"role": "user", "content": prompt}]

    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            input_ids,
            max_new_tokens=1,
            do_sample=False,
            output_scores=True,
            return_dict_in_generate=True,
        )

    logits = out.scores[0][0]
    probs = F.softmax(logits, dim=-1)

    p_yes = probs[yes_id].item()
    p_no = probs[no_id].item()

    if (p_yes + p_no) < 1e-6:
        print(f"Warning: near-zero Yes/No mass for '{obj}' ({category}). Top tokens:")
        top = torch.topk(probs, 10)
        for val, idx in zip(top.values, top.indices):
            print(f"  {tokenizer.decode([idx])!r}: {val.item():.4f}")
        return None

    typicality_prob = p_yes / (p_yes + p_no)
    return typicality_prob


Yes No


In [21]:
# Reset column in case a previous buggy run stored prompt text instead of scores
df["llama_typicality"] = pd.to_numeric(df["llama_typicality"], errors="coerce") if "llama_typicality" in df.columns else None

output_path = f"combined_prototypes_{LANGUAGE}.csv"

for idx, row in df.iterrows():
    if pd.notna(df.at[idx, "llama_typicality"]):
        continue  # already done, skip (useful on resume)
    score = get_typicality(row[instance_col], row["category"])
    df.at[idx, "llama_typicality"] = score
    df.to_csv(output_path, index=False)


In [22]:
df.to_csv(f"gemma_{LANGUAGE.lower()}_logprob_prototypical.csv", index=False)
